# 02 — Scanpy CPU Harmonized: Mouse Brain 1.3M

**Run on Libra CPU. Estimated time: 1-2 hours.**

In [ ]:
import json, time, os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KNeighborsTransformer

print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

In [ ]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"]["mouse_brain_1m"]
prefix = dcfg["pipeline_prefix"]
timings = {}

In [ ]:
t0 = time.time()
adata = sc.read_h5ad(dcfg["canonical_h5ad"])
timings["load"] = time.time() - t0
print(f"Loaded in {timings['load']:.1f}s: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
assert "counts" in adata.layers, "Missing counts layer — run 01_prepare first"

## Normalize + log1p

In [ ]:
t0 = time.time()
sc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
sc.pp.log1p(adata)
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

## HVG

In [ ]:
t0 = time.time()
sc.pp.highly_variable_genes(
    adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
    flavor="seurat_v3", subset=False, inplace=True,
)
timings["hvg"] = time.time() - t0
print(f"HVG ({adata.var['highly_variable'].sum()} genes): {timings['hvg']:.1f}s")

## PCA

In [ ]:
t0 = time.time()
sc.pp.pca(
    adata, n_comps=gcfg["pca_n_comps"], svd_solver="arpack",
    random_state=gcfg["random_state"], mask_var="highly_variable",
)
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

## Neighbors

In [ ]:
t0 = time.time()
transformer = KNeighborsTransformer(
    n_neighbors=dcfg["n_neighbors"], mode="distance",
    metric=gcfg["neighbor_metric"], algorithm="brute",
)
sc.pp.neighbors(
    adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca", method=gcfg["neighbor_method"],
    transformer=transformer, metric=gcfg["neighbor_metric"],
    random_state=gcfg["random_state"],
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors: {timings['neighbors']:.1f}s")

## Leiden

In [ ]:
t0 = time.time()
leiden_kwargs = {
    "resolution": dcfg["leiden_resolution"],
    "flavor": gcfg["scanpy_leiden_flavor"],
    "random_state": gcfg["random_state"],
    "key_added": "leiden",
}
if gcfg.get("scanpy_leiden_n_iterations") is not None:
    leiden_kwargs["n_iterations"] = gcfg["scanpy_leiden_n_iterations"]
sc.tl.leiden(adata, **leiden_kwargs)
timings["leiden"] = time.time() - t0
n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

## UMAP

In [ ]:
t0 = time.time()
sc.tl.umap(
    adata, min_dist=gcfg["umap_min_dist"], spread=gcfg["umap_spread"],
    init_pos="spectral", random_state=gcfg["random_state"],
)
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

## Differential Expression

In [ ]:
t0 = time.time()
sc.tl.rank_genes_groups(
    adata, groupby="leiden", method=gcfg["de_method"],
    corr_method=gcfg["de_corr_method"], use_raw=False, pts=True,
)
timings["de"] = time.time() - t0
print(f"DE: {timings['de']:.1f}s")

## Save outputs

In [ ]:
out = f"write/{prefix}_scanpy_cpu_harmonized"

# Clusters
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)

# Markers
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(f"{out}_markers.csv", index=False)
markers_filt = markers[(markers["pvals_adj"] < 0.05) & (markers["logfoldchanges"] > 0.1)]
markers_filt.to_csv(f"{out}_markers_filtered.csv", index=False)

# UMAP
pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names.astype(str),
    columns=["UMAP_1", "UMAP_2"],
).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

# HVG
adata.var.loc[adata.var["highly_variable"], []].reset_index(names="gene").to_csv(f"{out}_hvg.csv", index=False)

# h5ad
t0 = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0

# Spec
spec = {
    "pipeline": "scanpy_cpu_harmonized",
    "dataset": dcfg["name"],
    "scanpy_version": sc.__version__,
    "timings_seconds": timings,
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": int(adata.var["highly_variable"].sum()),
        "n_clusters": n_clusters,
        "n_marker_rows": len(markers),
        "n_marker_rows_filtered": len(markers_filt),
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nClusters: {n_clusters}")
print(f"Filtered markers: {len(markers_filt)}")